# ML Model Training

Dataset: IEEE-CIS Fraud Detection

| Model | Model Type | Notes |
|---------|------------|-------|
| 1 | LightGBM | Baseline features, binary classification, scale_pos_weight, early stopping |
| 2 | LightGBM | Engineered features, binary classification, scale_pos_weight, early stopping |
| 3 | Isolation Forest | Scaled data, contamination=0.035, anomaly detection |
| 4 | Local Outlier Factor | Scaled data, sub-sampled training (50k), novelty=True, contamination=0.035 |
| 5 | XGBoost | Engineered features, binary logistic, scale_pos_weight, early stopping, tree_method=hist |

In [1]:
import pandas as pd
import numpy as np
import gc
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import average_precision_score, roc_auc_score

# ---------------------------------------------------------
# HELPER: encode any remaining object columns
# Called on each split after loading from parquet, in case
# the parquet was saved before the label encoding fix was applied
# ---------------------------------------------------------
def encode_object_cols(df_tr, df_vl, df_te):
    obj_cols = [c for c in df_tr.columns if df_tr[c].dtype == object]
    if obj_cols:
        print(f"  Re-encoding {len(obj_cols)} object columns: {obj_cols[:5]}{'...' if len(obj_cols)>5 else ''}")
        for col in obj_cols:
            le = LabelEncoder()
            combined = pd.concat([df_tr[col], df_vl[col], df_te[col]],
                                  ignore_index=True).astype(str)
            le.fit(combined)
            df_tr[col] = le.transform(df_tr[col].astype(str))
            df_vl[col] = le.transform(df_vl[col].astype(str))
            df_te[col] = le.transform(df_te[col].astype(str))
    return df_tr, df_vl, df_te

# ---------------------------------------------------------
# 1. HELPER: TRAINING ENGINE
# ---------------------------------------------------------
def train_lgb(X_train, y_train, X_val, y_val, name="Model"):
    print(f"\n--- Training {name} ---")
    weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
    dtrain = lgb.Dataset(X_train, label=y_train)
    dval   = lgb.Dataset(X_val,   label=y_val, reference=dtrain)

    params = {
        'objective':        'binary',
        'metric':           'average_precision',
        'scale_pos_weight': weight,
        'learning_rate':    0.03,
        'num_leaves':       128,         # reduced from 256 to reduce overfitting
        'min_child_samples': 100,        # prevents splits on tiny fraud clusters
        'subsample':        0.8,         # row subsampling
        'colsample_bytree': 0.8,         # feature subsampling
        'reg_alpha':        0.1,         # L1 regularisation
        'verbosity':        -1
    }

    model = lgb.train(
        params, dtrain,
        valid_sets=[dval],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=100)]
    )

    probs = model.predict(X_val)
    print(f"  Val AUPRC: {average_precision_score(y_val, probs):.4f} | ROC-AUC: {roc_auc_score(y_val, probs):.4f}")
    return probs, roc_auc_score(y_val, probs), average_precision_score(y_val, probs), model


# ---------------------------------------------------------
# 2. FEATURE IMPACT ANALYSIS
# Requirement: run best classical model with and without engineered features and report AUPRC difference.
# Both comparisons must be on the SAME test set for consistency.
# ---------------------------------------------------------
print("=" * 60)
print("EXPERIMENT 1: Feature Impact Analysis")
print("=" * 60)

# --- Baseline ---
df_base_train = pd.read_parquet('data/iceee_baseline_train.parquet')
df_base_val   = pd.read_parquet('data/iceee_baseline_val.parquet')
df_base_test  = pd.read_parquet('data/iceee_baseline_test.parquet')

df_base_train, df_base_val, df_base_test = encode_object_cols(df_base_train, df_base_val, df_base_test)

X_tr_b = df_base_train.drop(['isFraud', 'TransactionID', 'TransactionDT'], axis=1, errors='ignore')
y_tr_b = df_base_train['isFraud']
X_vl_b = df_base_val.drop(['isFraud', 'TransactionID', 'TransactionDT'], axis=1, errors='ignore')
y_vl_b = df_base_val['isFraud']
X_te_b = df_base_test.drop(['isFraud', 'TransactionID', 'TransactionDT'], axis=1, errors='ignore')
y_te_b = df_base_test['isFraud']
del df_base_train, df_base_val, df_base_test; gc.collect()

_, _, _, lgb_base_model = train_lgb(X_tr_b, y_tr_b, X_vl_b, y_vl_b, "Baseline LGB")

# Score baseline on TEST set (not val — must match all other reported metrics)
probs_base_test = lgb_base_model.predict(X_te_b)
auprc_base_test = average_precision_score(y_te_b, probs_base_test)
auc_base_test   = roc_auc_score(y_te_b, probs_base_test)
print(f"Baseline TEST  — AUPRC: {auprc_base_test:.4f} | ROC-AUC: {auc_base_test:.4f}")
del X_tr_b, X_vl_b, X_te_b, y_tr_b, y_vl_b; gc.collect()

# --- Engineered ---
df_eng_train = pd.read_parquet('data/iceee_feature_train.parquet')
df_eng_val   = pd.read_parquet('data/iceee_feature_val.parquet')
df_eng_test  = pd.read_parquet('data/iceee_feature_test.parquet')

df_eng_train, df_eng_val, df_eng_test = encode_object_cols(df_eng_train, df_eng_val, df_eng_test)

X_tr_e = df_eng_train.drop(['isFraud', 'TransactionID', 'TransactionDT'], axis=1, errors='ignore')
y_tr_e = df_eng_train['isFraud']
X_vl_e = df_eng_val.drop(['isFraud', 'TransactionID', 'TransactionDT'], axis=1, errors='ignore')
y_vl_e = df_eng_val['isFraud']
X_te_e = df_eng_test.drop(['isFraud', 'TransactionID', 'TransactionDT'], axis=1, errors='ignore')
y_te_e = df_eng_test['isFraud']
del df_eng_train, df_eng_val, df_eng_test; gc.collect()

_, _, _, lgb_eng_model = train_lgb(X_tr_e, y_tr_e, X_vl_e, y_vl_e, "Engineered LGB")

# Score engineered on TEST set
probs_lgb_test  = lgb_eng_model.predict(X_te_e)
auprc_eng_test  = average_precision_score(y_te_e, probs_lgb_test)
auc_eng_test    = roc_auc_score(y_te_e, probs_lgb_test)
print(f"Engineered TEST — AUPRC: {auprc_eng_test:.4f} | ROC-AUC: {auc_eng_test:.4f}")
print(f"AUPRC delta (engineered - baseline): {auprc_eng_test - auprc_base_test:+.4f}")

# Save feature impact results (both on test set for consistency)
impact_df = pd.DataFrame({
    'Metric':     ['ROC-AUC', 'AUPRC'],
    'Baseline':   [auc_base_test,  auprc_base_test],
    'Engineered': [auc_eng_test,   auprc_eng_test]
})
impact_df.to_csv('results/feature_impact_results.csv', index=False)


# ---------------------------------------------------------
# 3. XGBOOST (on test set, val used only for early stopping)
# ---------------------------------------------------------
print("\n" + "=" * 60)
print("EXPERIMENT 2: XGBoost")
print("=" * 60)

dtrain_xgb = xgb.DMatrix(X_tr_e, label=y_tr_e)
dval_xgb   = xgb.DMatrix(X_vl_e, label=y_vl_e)
dtest_xgb  = xgb.DMatrix(X_te_e, label=y_te_e)

xgb_params = {
    'objective':        'binary:logistic',
    'eval_metric':      'aucpr',
    'scale_pos_weight': len(y_tr_e[y_tr_e==0]) / len(y_tr_e[y_tr_e==1]),
    'tree_method':      'hist',
    'learning_rate':    0.03,
    'max_depth':        6,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
}

clf_xgb = xgb.train(
    xgb_params, dtrain_xgb,
    num_boost_round=1000,
    evals=[(dval_xgb, 'valid')],
    early_stopping_rounds=100,   # increased from 50 — AUCPR signal is noisy on imbalanced data
    verbose_eval=100
)

probs_xgb = clf_xgb.predict(dtest_xgb)
print(f"XGBoost TEST — AUPRC: {average_precision_score(y_te_e, probs_xgb):.4f} | ROC-AUC: {roc_auc_score(y_te_e, probs_xgb):.4f}")


# ---------------------------------------------------------
# 4. CLASSICAL ANOMALY DETECTION
# Scores are inverted and normalized here so the results
# notebook does NOT need INVERT_MODELS logic
# ---------------------------------------------------------
print("\n" + "=" * 60)
print("EXPERIMENT 3: Classical Anomaly Detection")
print("=" * 60)
scaler = StandardScaler()

# Use train median (not -999) for scaling — consistent with FE fillna fixes
train_median = X_tr_e.median()
X_tr_s = scaler.fit_transform(X_tr_e.fillna(train_median))
X_te_s = scaler.transform(X_te_e.fillna(train_median))  # use train median on test too

X_tr_s = scaler.fit_transform(X_tr_e.fillna(train_median))
X_te_s = scaler.transform(X_te_e.fillna(train_median))

# Isolation Forest
print("Fitting Isolation Forest...")
iso = IsolationForest(n_estimators=100, contamination=0.035, random_state=42)
iso.fit(X_tr_s)
iso_scores = iso.decision_function(X_te_s)
# Invert: more negative score = more anomalous → flip so high = fraud
iso_probs = -iso_scores
iso_probs = (iso_probs - iso_probs.min()) / (iso_probs.max() - iso_probs.min())
print(f"ISO TEST — AUPRC: {average_precision_score(y_te_e, iso_probs):.4f} | ROC-AUC: {roc_auc_score(y_te_e, iso_probs):.4f}")

# LOF — subsampled due to memory constraints
print("Fitting LOF on 50k subsample...")
lof = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=0.035)
lof.fit(X_tr_s[:50000])
lof_scores = lof.decision_function(X_te_s)
# Invert: same logic as ISO
lof_probs = -lof_scores
lof_probs = (lof_probs - lof_probs.min()) / (lof_probs.max() - lof_probs.min())
print(f"LOF TEST — AUPRC: {average_precision_score(y_te_e, lof_probs):.4f} | ROC-AUC: {roc_auc_score(y_te_e, lof_probs):.4f}")


# ---------------------------------------------------------
# 5. EXPORT
# ---------------------------------------------------------
final_probs = pd.DataFrame({
    'actual': y_te_e,
    'lgb':    probs_lgb_test,
    'xgb':    probs_xgb,
    'iso':    iso_probs,          # already inverted and normalized
    'lof':    lof_probs           # already inverted and normalized
})
final_probs.to_csv('results/all_ml_probs.csv', index=False)

print("\n" + "=" * 60)
print("All ML experiments complete.")
print(f"  Feature impact saved: results/feature_impact_results.csv")
print(f"  Model probs saved:    results/all_ml_probs.csv")
print("=" * 60)

EXPERIMENT 1: Feature Impact Analysis
  Re-encoding 28 object columns: ['card4', 'card6', 'M1', 'M2', 'M3']...

--- Training Baseline LGB ---
Training until validation scores don't improve for 100 rounds
[100]	valid_0's average_precision: 0.446778
[200]	valid_0's average_precision: 0.511046
[300]	valid_0's average_precision: 0.537438
[400]	valid_0's average_precision: 0.549854
[500]	valid_0's average_precision: 0.557255
[600]	valid_0's average_precision: 0.561234
[700]	valid_0's average_precision: 0.566147
[800]	valid_0's average_precision: 0.569424
[900]	valid_0's average_precision: 0.573712
[1000]	valid_0's average_precision: 0.576853
Did not meet early stopping. Best iteration is:
[999]	valid_0's average_precision: 0.576904
  Val AUPRC: 0.5769 | ROC-AUC: 0.9187
Baseline TEST  — AUPRC: 0.5442 | ROC-AUC: 0.8949

--- Training Engineered LGB ---
Training until validation scores don't improve for 100 rounds
[100]	valid_0's average_precision: 0.472473
[200]	valid_0's average_precision: 0.